In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../data/raw/Dataset.csv")

df = pd.read_csv(RAW_PATH)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Rows: 8790, Columns: 10


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate show_id:", df["show_id"].duplicated().sum())

missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing

Duplicate rows: 0
Duplicate show_id: 0


,missing_count,missing_pct
show_id,0,0.0
type,0,0.0
title,0,0.0
director,0,0.0
country,0,0.0
date_added,0,0.0
release_year,0,0.0
rating,0,0.0
duration,0,0.0
listed_in,0,0.0


In [4]:
placeholders = {"", "unknown", "n/a", "na", "none", "null", "nan", "not given"}

for col in df.select_dtypes(include="object").columns:
    n_hidden = df[col].astype(str).str.strip().str.lower().isin(placeholders).sum()
    n_spaces = (df[col].astype(str) != df[col].astype(str).str.strip()).sum()
    print(f"{col:<12} hidden missing: {n_hidden:<5} extra whitespace: {n_spaces}")

show_id      hidden missing: 0     extra whitespace: 0
type         hidden missing: 0     extra whitespace: 0
title        hidden missing: 1     extra whitespace: 1
director     hidden missing: 2588  extra whitespace: 0
country      hidden missing: 287   extra whitespace: 0
date_added   hidden missing: 0     extra whitespace: 0
rating       hidden missing: 0     extra whitespace: 0
duration     hidden missing: 0     extra whitespace: 0
listed_in    hidden missing: 0     extra whitespace: 0


C:\Users\HP\AppData\Local\Temp\ipykernel_2496\1569269845.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [5]:
print(df["type"].value_counts(), end="\n\n")
print(df["rating"].value_counts(), end="\n\n")
print(df["country"].value_counts().head(8), end="\n\n")
print(df["director"].value_counts().head(5), end="\n\n")
print(df["date_added"].head(10).tolist())
print(df["duration"].str.extract(r"([A-Za-z]+)")[0].value_counts())

type
Movie      6126
TV Show    2664
Name: count, dtype: int64

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

country
United States     3240
India             1057
United Kingdom     638
Pakistan           421
Not Given          287
Canada             271
Japan              259
South Korea        214
Name: count, dtype: int64

director
Not Given                 2588
Rajiv Chilaka               20
Raúl Campos, Jan Suter      18
Alastair Fothergill         18
Suhas Kadav                 16
Name: count, dtype: int64

['9/25/2021', '9/24/2021', '9/24/2021', '9/22/2021', '9/24/2021', '9/24/2021', '9/24/2021', '5/1/2021', '9/23/2021', '5/1/2021']
0
min        6126
Season     1791
Seasons     873
Name: count, dtype: int64


In [6]:
def clean_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of the Netflix dataset."""
    df = raw.copy()

    # 1. Strip whitespace in all text columns
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].str.strip()

    # 2. Replace placeholder text with real missing values, then label them
    df[["director", "country"]] = (
        df[["director", "country"]].replace("Not Given", pd.NA).fillna("Unknown")
    )

    # 3. Dates: "9/25/2021" -> datetime, plus year and month features
    df["date_added"] = pd.to_datetime(df["date_added"], format="%m/%d/%Y", errors="coerce")
    df["year_added"] = df["date_added"].dt.year
    df["month_added"] = df["date_added"].dt.month

    # 4. Duration: "90 min" -> 90 / "min", "2 Seasons" -> 2 / "Season"
    parts = df["duration"].str.extract(r"(?P<value>\d+)\s*(?P<unit>[A-Za-z]+)")
    df["duration_value"] = pd.to_numeric(parts["value"])
    df["duration_unit"] = parts["unit"].replace({"Seasons": "Season"})

    # 5. Categories and helper columns
    df["type"] = df["type"].astype("category")
    df["rating"] = df["rating"].astype("category")
    df["primary_country"] = df["country"].str.split(",").str[0].str.strip()
    df["genres"] = df["listed_in"].str.split(",").apply(lambda xs: [x.strip() for x in xs])

    return df


clean_df = clean_data(df)
clean_df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_value,duration_unit,primary_country,genres
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,90,min,United States,[Documentaries]
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,1,Season,France,"[Crime TV Shows, International TV Shows, TV Ac..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,1,Season,United States,"[TV Dramas, TV Horror, TV Mysteries]"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,91,min,Brazil,"[Children & Family Movies, Comedies]"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,125,min,United States,"[Dramas, Independent Movies, International Mov..."


In [7]:
print("Missing values:\n", clean_df.isna().sum(), end="\n\n")
print("Unknown director:", (clean_df["director"] == "Unknown").sum())
print("Unknown country:", (clean_df["country"] == "Unknown").sum(), end="\n\n")
print(clean_df["duration_unit"].value_counts(), end="\n\n")
print(clean_df[["date_added", "year_added", "duration_value"]].describe())
clean_df.dtypes

Missing values:
 show_id            0
type               0
title              0
director           0
country            0
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
year_added         0
month_added        0
duration_value     0
duration_unit      0
primary_country    0
genres             0
dtype: int64

Unknown director: 2588
Unknown country: 287

duration_unit
min       6126
Season    2664
Name: count, dtype: int64

                       date_added   year_added  duration_value
count                        8790  8790.000000     8790.000000
mean   2019-05-17 21:44:01.638225  2018.873606       69.934471
min           2008-01-01 00:00:00  2008.000000        1.000000
25%           2018-04-06 00:00:00  2018.000000        2.000000
50%           2019-07-03 00:00:00  2019.000000       88.500000
75%           2020-08-19 18:00:00  2020.000000      106.000000
max           2021-09-25 00:00:00  2021.000000      312.000000
std          

show_id                       str
type                     category
title                         str
director                      str
country                       str
date_added         datetime64[us]
release_year                int64
rating                   category
duration                      str
listed_in                     str
year_added                  int32
month_added                 int32
duration_value              int64
duration_unit                 str
primary_country            object
genres                     object
dtype: object

In [8]:
CLEAN_PATH = Path("../data/processed/netflix_clean.csv")
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

clean_df.drop(columns=["genres"]).to_csv(CLEAN_PATH, index=False)

# Reload to prove the saved file is valid
check = pd.read_csv(CLEAN_PATH, parse_dates=["date_added"])
print("Saved shape:", check.shape)
print("Date column type:", check["date_added"].dtype)
print("Missing values:", int(check.isna().sum().sum()))

Saved shape: (8790, 15)
Date column type: datetime64[us]
Missing values: 0


## Summary of cleaning steps

| Issue found | Action |
|---|---|
| No NaN values or duplicate rows | Verified with `isna()` and `duplicated()` |
| "Not Given" placeholders in `director` (2,588) and `country` (287) | Replaced with `Unknown` |
| Extra whitespace in text (e.g. one title) | Stripped from all text columns |
| `date_added` stored as text (`9/25/2021`) | Converted to datetime; added `year_added`, `month_added` |
| `duration` mixed minutes and seasons | Split into `duration_value` and `duration_unit` |
| Categorical columns | `type` and `rating` converted to category |
| Helper features | Added `primary_country` and `genres` for later tasks |

**Result:** a clean dataset of 8,790 rows saved to `data/processed/netflix_clean.csv`.